<a href="https://colab.research.google.com/github/fdx-hw/cosc-650/blob/week_7_adaptation/week7_adaptation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7: The Adaptation Decision and a Production Dataset

**Task:** support-ticket triage -- route a short ticket to one of five labels (`billing`,
`account_access`, `bug_report`, `feature_request`, `cancellation`). No GPU, no fine-tuning:
the adaptation method is retrieval-selected (dynamic) few-shot, compared against zero-shot.

**Data:** synthetic, built from templates. That makes it easy to
audit and reproduce, but it has a real weakness I'm designing around instead of ignoring:
template data means eval tickets can sound a lot like training tickets, which flatters
few-shot. So some eval templates are held out of training entirely, and two "hard" kinds of
ticket are tagged before any model sees them.

**Model:** I used Claude (`claude-haiku-4-5-20251001`)
instead, the same swap already disclosed in Weeks 5-6, with the same Colab-safe key handling.
Calls pin `temperature` to 0 (the run prints a note if the API refuses that), and each (variant, ticket) pair is called exactly once and cached,
so zero-shot and few-shot are compared on one fixed set of outputs.

After the first run showed both methods near ceiling, I added a clearly-labeled post-hoc stress
test in Part 3 (it is not part of the 64-ticket eval).



In [ ]:
%pip install -q tiktoken sentence-transformers faiss-cpu anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.2 MB/s eta 0:00:00


## Part 1: Prepare the dataset

**Labeling rule:** a ticket gets the label for the *primary thing the
customer is asking us to do*. A charge dispute or refund is `billing`; ending the subscription
is `cancellation`; something broken is `bug_report`; something missing they want built is
`feature_request`; not being able to get into the account is `account_access`.

Two hard kinds are generated on purpose and tagged with `kind`:
- `convention`: tickets that mention both a charge and a cancellation, where the label depends
  on which action they ask for (e.g. "I cancelled but was still charged" is `billing`).
- `lookalike`: tickets that share topic words with a *different* label (e.g. "the export
  needs CSV" is a feature request, "the export is missing the header" is a bug).

Everything else is `standard`. Each label also has one held-out template used only in eval
(`seen_template=False`), to see how few-shot behaves on phrasing it has no near-copy of.
The raw pool also gets some deliberately injected noise (empty text, bad labels, duplicates,
conflicting labels, overlong text) so the cleaning step has something real to do -- the noise
is synthetic, not something I found in the data.


In [ ]:
import os, re, json, random, hashlib, pathlib, time, math, itertools
from collections import Counter, defaultdict

SEED = 7
rng = random.Random(SEED)

LABELS = ['billing', 'account_access', 'bug_report', 'feature_request', 'cancellation']
VALID = set(LABELS)
MAX_CHARS = 280

SLOT = {
    'plan':    ['Pro plan', 'Team plan', 'Starter plan', 'annual plan', 'Business plan'],
    'amount':  ['$9.99', '$29', '$49.99', '$120', '$15'],
    'when':    ['yesterday', 'this morning', 'last week', 'three days ago', 'twice this month'],
    'since':   ['yesterday', 'this morning', 'last week', 'Monday', 'the 3rd'],
    'feature': ['export', 'dashboard', 'mobile app', 'search', 'report builder', 'calendar sync', 'notifications'],
    'x':       ['CSV export', 'dark mode', 'keyboard shortcuts', 'bulk editing', 'a weekly summary email', 'custom filters', 'PDF output'],
    'error':   ['error 500', 'a blank screen', 'a timeout', 'a crash', 'duplicate entries', 'a spinning loader'],
    'gadget':  ['my phone', 'my tablet', 'my security key', 'my old laptop'],
    'device':  ['my phone', 'my laptop', 'the tablet', 'Safari', 'Chrome', 'the Android app'],
    'reason':  ['because I no longer need it', 'because we are switching tools', 'because the price is too high',
                'because I am moving abroad', 'because our budget changed'],
}

TEMPLATES = [
    # billing
    ('b1', 'billing', "I was charged {amount} {when} but I'm on the {plan}", 'standard', False),
    ('b2', 'billing', "Why was I billed twice for the {plan}?", 'standard', False),
    ('b3', 'billing', "Please refund the {amount} charge from {since}", 'standard', False),
    ('b4', 'billing', "My invoice shows the wrong amount, it should be {amount}", 'standard', False),
    ('b5', 'billing', "I need a receipt for my {plan} payment {when}", 'standard', False),
    ('b6', 'billing', "Can you update the card on file for my {plan}?", 'standard', False),
    ('b7', 'billing', "There is an unexpected {amount} line on my statement from {since}", 'standard', True),
    # account_access
    ('a1', 'account_access', "I can't log in to my account on {device} since {since}", 'standard', False),
    ('a2', 'account_access', "The password reset email never arrives on {device}, I asked for it {when}", 'standard', False),
    ('a3', 'account_access', "My account has been locked on {device} since {since}", 'standard', False),
    ('a4', 'account_access', "I lost {gadget} {when} and can't pass two-factor sign in", 'standard', False),
    ('a5', 'account_access', "It says my account does not exist when I sign in on {device} since {since}", 'standard', False),
    ('a6', 'account_access', "I never got the verification code {when} on {device}", 'standard', False),
    ('a7', 'account_access', "Locked out again, the login keeps rejecting me on {device}", 'standard', True),
    # bug_report
    ('g1', 'bug_report', "The {feature} shows {error} when I click it", 'standard', False),
    ('g2', 'bug_report', "The {feature} stopped working after the update and shows {error}", 'standard', False),
    ('g3', 'bug_report', "The app crashes whenever I open the {feature}", 'standard', False),
    ('g4', 'bug_report', "The {feature} keeps loading forever and then shows {error}", 'standard', False),
    ('g5', 'bug_report', "Clicking save in the {feature} gives me {error}", 'standard', False),
    ('g6', 'bug_report', "I'm seeing {error} on the {feature} since the latest release", 'standard', False),
    ('g7', 'bug_report', "Something is broken in the {feature}, nothing happens after I tap it on {device}", 'standard', True),
    # feature_request
    ('f1', 'feature_request', "Please add {x} to the {feature}", 'standard', False),
    ('f2', 'feature_request', "Can you add {x} to the {feature}? I would use it every day", 'standard', False),
    ('f3', 'feature_request', "I'd love {x} in the {feature}", 'standard', False),
    ('f4', 'feature_request', "Feature idea: {x} for the {feature}", 'standard', False),
    ('f5', 'feature_request', "It would be great if the {feature} supported {x}", 'standard', False),
    ('f6', 'feature_request', "Any plans to bring {x} to the {feature}?", 'standard', False),
    ('f7', 'feature_request', "Wishlist item from our team: {x}, ideally inside the {feature}", 'standard', True),
    # cancellation
    ('k1', 'cancellation', "I want to cancel my {plan} {reason}", 'standard', False),
    ('k2', 'cancellation', "Please close my account and stop all future renewals {reason}", 'standard', False),
    ('k3', 'cancellation', "How do I cancel my {plan} before the next renewal?", 'standard', False),
    ('k4', 'cancellation', "Cancel my {plan} effective immediately {reason}", 'standard', False),
    ('k5', 'cancellation', "I no longer need the service, please end my {plan}", 'standard', False),
    ('k6', 'cancellation', "Stop my subscription at the end of this billing period {reason}", 'standard', False),
    ('k7', 'cancellation', "We are shutting down the project, so please terminate our {plan}", 'standard', True),
    # hard: convention (charge vs cancel -- label = the primary ask)
    ('hc1', 'billing', "I cancelled my {plan} {when} but I was still charged {amount}", 'convention', False),
    ('hc2', 'billing', "Charged {amount} after I closed my account, please refund it", 'convention', False),
    ('hc3', 'billing', "Why was I billed for the {plan} when I asked to cancel {when}?", 'convention', False),
    ('hc4', 'cancellation', "I keep getting charged {amount}, so please cancel my {plan}", 'convention', False),
    ('hc5', 'cancellation', "The price is too high, cancel my {plan} before it renews", 'convention', False),
    ('hc6', 'billing', "Refund me for the renewal, I had already cancelled {when}", 'convention', True),
    # hard: lookalike (shares topic words with a different label)
    ('hl1', 'feature_request', "The {feature} needs {x}, right now it can't do that", 'lookalike', False),
    ('hl2', 'bug_report', "The {feature} is missing {x} in the output, it used to be there", 'lookalike', False),
    ('hl3', 'bug_report', "The sign in page crashes with {error} on {device}", 'lookalike', False),
    ('hl4', 'account_access', "I can't sign in on {device} because my password was rejected", 'lookalike', False),
    ('hl5', 'feature_request', "Add a way to reset my password from the {feature}", 'lookalike', False),
    ('hl6', 'feature_request', "Add an option to pause billing instead of cancelling my {plan}", 'lookalike', True),
]

def cores_for(tmpl):
    """Every distinct slot fill of one template, shuffled."""
    names = list(dict.fromkeys(re.findall(r'\{(\w+)\}', tmpl)))
    out = [tmpl.format(**dict(zip(names, combo))) for combo in itertools.product(*[SLOT[n] for n in names])]
    rng.shuffle(out)
    return out

def take_round_robin(pools, n):
    """pools = [(template_id, cores)]; take one core per template in turn until n are taken."""
    pools = [(tid, list(c)) for tid, c in pools]
    taken = []
    while len(taken) < n and any(c for _, c in pools):
        for tid, c in pools:
            if c and len(taken) < n:
                taken.append((tid, c.pop(0)))
    return taken

OPEN = ['', '', 'Hi, ', 'Hello, ', 'Hey team, ']
TAIL = ['', '', '', ' Thanks!', ' Please help.', ' This is urgent.', ' Any update?']
def decorate(core):
    o, t = rng.choice(OPEN), rng.choice(TAIL)
    if o and not re.match(r"I\b", core):
        core = core[0].lower() + core[1:]
    return o + core + t

def make(label, tid, kind, held_out, core, split):
    return {'text': decorate(core), 'label': label, 'template': tid, 'kind': kind,
            'seen_template': not held_out, 'split': split}

raw = []
for lab in LABELS:
    std = [t for t in TEMPLATES if t[1] == lab and t[3] == 'standard']
    seen = [(t[0], cores_for(t[2])) for t in std if not t[4]]
    taken = take_round_robin(seen, 50)
    rng.shuffle(taken)
    for i, (tid, core) in enumerate(taken):
        raw.append(make(lab, tid, 'standard', False, core, 'train' if i < 44 else 'eval'))
    held = [(t[0], cores_for(t[2])) for t in std if t[4]]
    for tid, core in take_round_robin(held, 4):
        raw.append(make(lab, tid, 'standard', True, core, 'eval'))
for tid, lab, tmpl, kind, held_out in TEMPLATES:
    if kind == 'standard':
        continue
    cores = cores_for(tmpl)
    if held_out:
        for core in cores[:2]:
            raw.append(make(lab, tid, kind, True, core, 'eval'))
    else:
        for core in cores[:3]:
            raw.append(make(lab, tid, kind, False, core, 'train'))
        for core in cores[3:4]:
            raw.append(make(lab, tid, kind, False, core, 'eval'))
n_planned = len(raw)

src = [r for r in raw if r['split'] == 'train']
rng.shuffle(src)
def take(n):
    global src
    out, src = src[:n], src[n:]
    return out
def noise_rec(text, label):
    return {'text': text, 'label': label, 'template': 'noise', 'kind': 'noise', 'seen_template': False, 'split': 'train'}

a, b = take(2)
a['label'] = a['label'].capitalize() + ' '
b['label'] = b['label'].upper()
for r in take(3):
    r['text'] = '  ' + r['text'].replace(' ', '  ', 1) + '\n'
noise = []
for text, lab in zip(['', '   ', '', '\t'], rng.sample(LABELS, 4)):
    noise.append(noise_rec(text, lab))
for text, lab in [('hi', 'unknown'), ('thanks', 'unknown'), ('test', 'refund')]:
    noise.append(noise_rec(text, lab))
for r in take(6):
    noise.append(dict(r))
for r, mode in zip(take(6), ['upper', 'upper', 'upper', 'bang', 'bang', 'bang']):
    d = dict(r)
    d['text'] = d['text'].upper() if mode == 'upper' else d['text'] + '!!'
    noise.append(d)
for r in take(2):
    d = dict(r)
    d['label'] = [l for l in LABELS if l != r['label']][0]
    noise.append(d)
for r in take(2):
    d = dict(r)
    d['text'] = ' '.join([d['text']] * 12)
    noise.append(d)
raw = raw + noise
print(f'planned examples: {n_planned}   injected noise records: {len(noise)}   raw pool: {len(raw)}')

planned examples: 314   injected noise records: 23   raw pool: 337


In [ ]:
def norm_key(text):
    return re.sub(r'\s+', ' ', re.sub(r'[^\w\s]', '', text.lower())).strip()

def clean(records):
    rep = Counter()
    kept = []
    for r in records:
        r = dict(r)
        t = re.sub(r'\s+', ' ', r['text']).strip() if isinstance(r['text'], str) else ''
        l = r['label'].strip().lower() if isinstance(r['label'], str) else ''
        if not t:
            rep['dropped: empty text'] += 1; continue
        if l not in VALID:
            rep['dropped: label not in the 5-label set'] += 1; continue
        if len(t) > MAX_CHARS:
            rep[f'dropped: longer than {MAX_CHARS} chars'] += 1; continue
        if t != r['text']:
            rep['repaired: whitespace'] += 1
        if l != r['label']:
            rep['repaired: label case/whitespace'] += 1
        r['text'], r['label'] = t, l
        kept.append(r)
    groups = defaultdict(list)
    for r in kept:
        groups[norm_key(r['text'])].append(r)
    out = []
    for grp in groups.values():
        if len({g['label'] for g in grp}) > 1:
            rep['dropped: same text, conflicting labels (all copies)'] += len(grp); continue
        out.append(grp[0])
        rep['dropped: exact/near duplicate'] += len(grp) - 1
    return out, rep

clean_records, CLEAN_REPORT = clean(raw)
train = [r for r in clean_records if r['split'] == 'train']
eval_ = [r for r in clean_records if r['split'] == 'eval']
rng.shuffle(train); rng.shuffle(eval_)

print(f'raw {len(raw)} -> clean {len(clean_records)}   (train {len(train)}, eval {len(eval_)})')
for k, v in sorted(CLEAN_REPORT.items()):
    print(f'  {k:55s} {v}')
overlap = {norm_key(r['text']) for r in train} & {norm_key(r['text']) for r in eval_}
print(f'train/eval overlap after cleaning (normalized text): {len(overlap)}')
print('requirement >=200 train:', 'PASS' if len(train) >= 200 else 'FAIL', '|',
      'requirement >=50 eval:', 'PASS' if len(eval_) >= 50 else 'FAIL')

raw 337 -> clean 312   (train 248, eval 64)
  dropped: empty text                                     4
  dropped: exact/near duplicate                           12
  dropped: label not in the 5-label set                   3
  dropped: longer than 280 chars                          2
  dropped: same text, conflicting labels (all copies)     4
  repaired: label case/whitespace                         2
  repaired: whitespace                                    3
train/eval overlap after cleaning (normalized text): 0
requirement >=200 train: PASS | requirement >=50 eval: PASS


In [ ]:
# ---- statistics: token length, label balance, composition, format checks ----
try:
    import tiktoken
    os.environ['TIKTOKEN_CACHE_DIR'] = str((pathlib.Path('.') / '.tiktoken_cache').resolve())
    _enc = tiktoken.get_encoding('cl100k_base')
    tok_len = lambda s: len(_enc.encode(s))
    TOK_NAME = 'tiktoken cl100k_base'
except Exception as e:
    tok_len = lambda s: len(re.findall(r"\w+|[^\w\s]", s))
    TOK_NAME = f'regex word/punctuation count (tiktoken unavailable: {type(e).__name__})'
print('token counter:', TOK_NAME, '(an approximation -- Claude uses its own tokenizer)')

def describe(vals):
    s = sorted(vals); n = len(s)
    q = lambda p: s[min(n - 1, int(p * n))]
    return {'n': n, 'min': s[0], 'p25': q(.25), 'median': q(.5), 'p75': q(.75), 'p90': q(.9),
            'max': s[-1], 'mean': round(sum(s) / n, 1)}

TOKEN_STATS = {}
for name, ds in (('train', train), ('eval', eval_)):
    lens = [tok_len(r['text']) for r in ds]
    TOKEN_STATS[name] = describe(lens)
    print(f'{name:5s} token length', TOKEN_STATS[name])
lens = [tok_len(r['text']) for r in train]
buckets = Counter((l // 4) * 4 for l in lens)
peak = max(buckets.values())
print('train token-length histogram (4-token buckets):')
for b in sorted(buckets):
    print(f'  {b:3d}-{b + 3:<3d} | ' + '#' * max(1, round(buckets[b] / peak * 40)) + f' {buckets[b]}')

print()
BALANCE = {}
for name, ds in (('train', train), ('eval', eval_)):
    c = Counter(r['label'] for r in ds)
    BALANCE[name] = dict(c)
    print(f'{name:5s} label balance {dict(c)}   max/min = {max(c.values()) / min(c.values()):.2f}')
print()
for name, ds in (('train', train), ('eval', eval_)):
    comp = Counter((r['kind'], 'seen template' if r['seen_template'] else 'held-out template') for r in ds)
    print(f'{name:5s} composition:', dict(comp))

print()
CHECKS = {
    'has text + label strings': lambda r: isinstance(r.get('text'), str) and isinstance(r.get('label'), str),
    'text non-empty': lambda r: bool(r['text'].strip()),
    'no leading/trailing whitespace': lambda r: r['text'] == r['text'].strip(),
    'no double spaces / newlines / tabs': lambda r: not re.search(r'\s{2,}|[\n\t]', r['text']),
    'label in the 5-label set': lambda r: r['label'] in VALID,
    f'length <= {MAX_CHARS} chars': lambda r: len(r['text']) <= MAX_CHARS,
    'printable ASCII only': lambda r: all(32 <= ord(ch) < 127 for ch in r['text']),
}
FORMAT_CHECKS = {}
allrows = train + eval_
for cname, fn in CHECKS.items():
    ok = sum(1 for r in allrows if fn(r))
    FORMAT_CHECKS[cname] = f'{ok}/{len(allrows)}'
    print(f'format check: {cname:38s} {ok}/{len(allrows)}')

token counter: tiktoken cl100k_base (an approximation -- Claude uses its own tokenizer)
train token length {'n': 248, 'min': 7, 'p25': 13, 'median': 15, 'p75': 17, 'p90': 20, 'max': 25, 'mean': 15.2}
eval  token length {'n': 64, 'min': 10, 'p25': 14, 'median': 16, 'p75': 18, 'p90': 20, 'max': 22, 'mean': 16.0}
train token-length histogram (4-token buckets):
    4-7   | # 1
    8-11  | ############# 34
   12-15  | ######################################## 103
   16-19  | ################################# 84
   20-23  | ######### 23
   24-27  | # 3

train label balance {'cancellation': 50, 'billing': 53, 'feature_request': 49, 'bug_report': 50, 'account_access': 46}   max/min = 1.15
eval  label balance {'account_access': 11, 'feature_request': 14, 'bug_report': 12, 'billing': 15, 'cancellation': 12}   max/min = 1.36

train composition: {('standard', 'seen template'): 218, ('convention', 'seen template'): 15, ('lookalike', 'seen template'): 15}
eval  composition: {('lookalike', 'seen templ

In [ ]:
os.makedirs('data', exist_ok=True)
def dump(path, rows):
    with open(path, 'w') as f:
        for r in rows:
            f.write(json.dumps({k: r[k] for k in ('text', 'label', 'kind', 'template', 'seen_template')}, sort_keys=True) + '\n')
dump('data/train.jsonl', train)
dump('data/eval.jsonl', eval_)
FINGERPRINT = hashlib.sha256(open('data/train.jsonl', 'rb').read() + open('data/eval.jsonl', 'rb').read()).hexdigest()[:16]
QUALITY = {
    'task': 'support-ticket triage (5 labels)', 'seed': SEED,
    'raw_pool': len(raw), 'clean': len(clean_records), 'train': len(train), 'eval': len(eval_),
    'cleaning': dict(CLEAN_REPORT), 'train_eval_text_overlap': len(overlap),
    'token_counter': TOK_NAME, 'token_stats': TOKEN_STATS, 'label_balance': BALANCE,
    'format_checks': FORMAT_CHECKS, 'dataset_fingerprint': FINGERPRINT,
}
with open('data/quality_report.json', 'w') as f:
    json.dump(QUALITY, f, indent=2)
print('wrote data/train.jsonl, data/eval.jsonl, data/quality_report.json')
print('dataset fingerprint:', FINGERPRINT)

wrote data/train.jsonl, data/eval.jsonl, data/quality_report.json
dataset fingerprint: 4a1f98496b0d7627


**Data-quality read-out.**

I generated 314 tickets and injected 23 bad records on purpose, so the raw pool was 337. The
cleaner dropped 4 empty texts, 3 invalid labels, 2 overlong tickets, and 12 exact or near
duplicates. It also dropped 4 records that were two pairs of the same text with conflicting
labels. It repaired 2 labels that only differed by case or spacing and 3 texts with stray
whitespace. That left 312 tickets: 248 train and 64 eval, with zero overlap between them.

The conflicting-label pairs are dropped whole because I can't tell which copy is right. That
cost two good training tickets, which is why train is 46-53 per label and not perfectly even
(max/min 1.15). Eval is less even, 11-15 per label, because I built it to cover the hard and
held-out groups, not to balance labels.

Tickets are short: median 15 tokens, p90 20, max 25 in train (tiktoken cl100k_base, which only
approximates Claude's tokenizer). All seven format checks pass on all 312 tickets. The dataset
fingerprint is `4a1f98496b0d7627`.

One caveat. The tickets come from templates, so eval tickets from seen templates are close
cousins of training tickets (median top-1 similarity 0.80, and 16% are at least 0.90). The
held-out-template and hard-ticket slices are the more honest read.


## Part 2: Adapt with dynamic few-shot (local selection)

Same retrieval stack as Week 5 -- `all-MiniLM-L6-v2` embeddings, a FAISS `IndexFlatIP` over
normalized vectors (inner product = cosine) -- but the "documents" are the training tickets.
For each eval ticket, the 4 most similar training tickets (with their labels) go into the
prompt, most similar first. Retrieval looks only at the ticket text, never at the eval label.


In [ ]:
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
K = 4

def embed(texts):
    return np.asarray(emb.encode(texts, normalize_embeddings=True)).astype('float32')

train_vecs = embed([r['text'] for r in train])
eval_vecs = embed([r['text'] for r in eval_])
index = faiss.IndexFlatIP(train_vecs.shape[1]); index.add(train_vecs)

def retrieve(vecs, k=K):
    sims, ids = index.search(vecs, k)
    return [([int(j) for j in ids[i]], [float(s) for s in sims[i]]) for i in range(len(ids))]

RETR = retrieve(eval_vecs, K)

def few_shot(text, k=K):
    ids, _ = retrieve(embed([text]), k)[0]
    return [train[j] for j in ids]

def knn_vote(ids):
    """Plurality label among retrieved neighbors; ties go to the most similar neighbor's label."""
    votes = Counter(train[j]['label'] for j in ids)
    top = max(votes.values())
    tied = {l for l, v in votes.items() if v == top}
    for j in ids:
        if train[j]['label'] in tied:
            return train[j]['label']

ex_i = next((i for i, r in enumerate(eval_) if r['kind'] == 'lookalike'), 0)
print('eval ticket:', eval_[ex_i]['text'], '(gold:', eval_[ex_i]['label'] + ')')
print('retrieved few-shot examples:')
for j, s in zip(*RETR[ex_i]):
    print(f'   {s:.2f}  {train[j]["text"]}  ->  {train[j]["label"]}')

top1 = sorted(s[0] for _, s in RETR)
print(f'\ntop-1 train similarity per eval ticket: median {top1[len(top1) // 2]:.2f}, '
      f'min {top1[0]:.2f}, max {top1[-1]:.2f}, share >= 0.90: {sum(s >= 0.90 for s in top1) / len(top1):.0%}')
pure = sum(all(train[j]['label'] == eval_[i]['label'] for j in RETR[i][0]) for i in range(len(eval_)))
print(f'eval tickets whose {K} neighbors ALL carry the gold label: {pure}/{len(eval_)}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

eval ticket: Hi, I can't sign in on Safari because my password was rejected Any update? (gold: account_access)
retrieved few-shot examples:
   0.74  Hello, my account has been locked on Safari since last week Any update?  ->  account_access
   0.71  I can't log in to my account on Safari since Monday This is urgent.  ->  account_access
   0.68  Hello, I can't sign in on Chrome because my password was rejected This is urgent.  ->  account_access
   0.67  Hey team, my account has been locked on Safari since Monday Please help.  ->  account_access

top-1 train similarity per eval ticket: median 0.80, min 0.45, max 0.99, share >= 0.90: 16%
eval tickets whose 4 neighbors ALL carry the gold label: 52/64


## Part 3: Evaluate, including one failure

Zero-shot and dynamic few-shot use the identical instruction line; the only difference is the
4 retrieved examples. Two extra rows cost no model call and anchor the comparison for the memo:
a kNN vote (predict the plurality label of the same 4 neighbors, no LLM at all) and the
majority-class floor. Predictions that don't parse to a valid label count as wrong and are
reported separately, never dropped.


In [ ]:
import getpass

def _get_api_key():
    """Colab secret, then env var, then an in-notebook prompt (same pattern as Weeks 5-6)."""
    try:
        from google.colab import userdata
        key = userdata.get('ANTHROPIC_API_KEY')
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get('ANTHROPIC_API_KEY')
    if key:
        return key
    try:
        return getpass.getpass('Enter your ANTHROPIC_API_KEY: ')
    except Exception:
        return None

MODEL = 'claude-haiku-4-5-20251001'
_api_key = _get_api_key()
_client = None
if _api_key:
    import anthropic
    _client = anthropic.Anthropic(api_key=_api_key, timeout=30.0, max_retries=1)
    print('API key found; client ready (30s timeout per call).')
else:
    print('[API-BLOCKED] no key found -- the dataset and retrieval above are real; the scored eval is skipped.')

def build_prompt(text, shots=None):
    head = ('Classify the support ticket into exactly one of these labels: ' + ', '.join(LABELS)
            + '.\nReply with only the label.\n\n')
    body = ''.join(f'Ticket: {s["text"]}\nLabel: {s["label"]}\n\n' for s in (shots or []))
    return head + body + f'Ticket: {text}\nLabel:'

def parse_label(raw):
    """-> (label, was_exact). 'invalid' if no valid label can be found in the reply."""
    if raw is None:
        return 'invalid', False
    s = raw.strip().lower()
    s = re.sub(r'^(intent|label|answer)\s*[:\-]\s*', '', s).strip(' .`*"\'\n')
    if s in VALID:
        return s, True
    hits = sorted((m.start(), lab) for lab in LABELS for m in re.finditer(rf'\b{lab}\b', raw.lower()))
    if hits:
        return hits[0][1], False
    return 'invalid', False

TEMP_PINNED = True      # flips to False (with a printed note) if the API refuses temperature for this model

def call_claude(prompt, retries=3):
    global TEMP_PINNED
    last = None
    for attempt in range(retries):
        try:
            t0 = time.time()
            kwargs = dict(model=MODEL, max_tokens=20, messages=[{'role': 'user', 'content': prompt}])
            if TEMP_PINNED:
                kwargs['extra_body'] = {'temperature': 0}
            resp = _client.messages.create(**kwargs)
            return {'raw': resp.content[0].text, 'in_tok': resp.usage.input_tokens,
                    'out_tok': resp.usage.output_tokens, 'secs': time.time() - t0, 'error': None}
        except Exception as e:
            last = f'{type(e).__name__}: {e}'
            if TEMP_PINNED and getattr(e, 'status_code', None) == 400 and 'temperature' in str(e).lower():
                TEMP_PINNED = False
                print('NOTE: the API refused temperature for this model -- continuing WITHOUT pinning it. '
                      'Outputs may vary slightly between runs; say so in the write-up.')
                continue
            if getattr(e, 'status_code', None) in (400, 401, 403, 404):
                break
            time.sleep(2 ** attempt)
    return {'raw': None, 'in_tok': 0, 'out_tok': 0, 'secs': 0.0, 'error': last}

CALLS = {}
def classify(variant, text, shots=None, retries=3):
    key = (variant, text)
    if key not in CALLS or CALLS[key]['error']:
        out = call_claude(build_prompt(text, shots), retries=retries)
        out['pred'], out['exact'] = parse_label(out['raw'])
        CALLS[key] = out
    return CALLS[key]



API key found; client ready (30s timeout per call).


In [ ]:
API_OK = False
if _client is not None:
    t0 = time.time()
    probe = classify('zero', eval_[0]['text'], retries=2)
    if probe['error']:
        print(f'[API-BLOCKED] test call failed after {time.time() - t0:.1f}s:\n  {probe["error"]}')
        print('Fix the key / model name / network above, then re-run this cell. Nothing else will be called.')
    else:
        API_OK = True
        print(f'test call OK in {time.time() - t0:.1f}s -> raw reply {probe["raw"]!r}, parsed {parse_label(probe["raw"])}')
        print('temperature pinned to 0:', TEMP_PINNED)


test call OK in 1.0s -> raw reply 'account_access', parsed ('account_access', True)
temperature pinned to 0: True


In [ ]:
ROWS = []
if API_OK:
    t_start = time.time()
    bad_streak = 0
    for i, r in enumerate(eval_):
        if i % 4 == 0:
            n_err = sum(1 for c in CALLS.values() if c['error'])
            print(f'  ticket {i}/{len(eval_)}   {time.time() - t_start:.0f}s elapsed   {n_err} failed calls so far', flush=True)
        ids, sims = RETR[i]
        shots = [train[j] for j in ids]
        z = classify('zero', r['text'])
        f = classify('few', r['text'], shots)
        bad_streak = bad_streak + 1 if (z['error'] or f['error']) else 0
        if bad_streak >= 3:
            print(f'stopping: 3 tickets in a row failed. Last error: {z["error"] or f["error"]}')
            break
        if z['error'] or f['error']:
            continue                       # never score an API failure as a wrong answer
        ROWS.append({
            'i': i, 'text': r['text'], 'gold': r['label'], 'kind': r['kind'],
            'seen_template': r['seen_template'],
            'zero': z['pred'], 'few': f['pred'], 'knn': knn_vote(ids),
            'shot_labels': [train[j]['label'] for j in ids], 'shot_texts': [train[j]['text'] for j in ids],
            'top1_sim': sims[0],
        })
    errs = sum(1 for c in CALLS.values() if c['error'])
    first = next((c['error'] for c in CALLS.values() if c['error']), None)
    if first:
        print('first API error:', first)
    print(f'{len(CALLS)} model calls made ({len(ROWS)} eval tickets x 2 variants), {errs} failed after retries')
    if len(ROWS) < len(eval_):
        print(f'WARNING: only {len(ROWS)}/{len(eval_)} tickets scored -- re-run this cell to retry the failed ones before reading any numbers.')
else:
    print('no scored run (see the test-call message above)')

  ticket 0/64   0s elapsed   0 failed calls so far
  ticket 4/64   5s elapsed   0 failed calls so far
  ticket 8/64   10s elapsed   0 failed calls so far
  ticket 12/64   15s elapsed   0 failed calls so far
  ticket 16/64   20s elapsed   0 failed calls so far
  ticket 20/64   25s elapsed   0 failed calls so far
  ticket 24/64   30s elapsed   0 failed calls so far
  ticket 28/64   35s elapsed   0 failed calls so far
  ticket 32/64   40s elapsed   0 failed calls so far
  ticket 36/64   45s elapsed   0 failed calls so far
  ticket 40/64   51s elapsed   0 failed calls so far
  ticket 44/64   56s elapsed   0 failed calls so far
  ticket 48/64   61s elapsed   0 failed calls so far
  ticket 52/64   66s elapsed   0 failed calls so far
  ticket 56/64   71s elapsed   0 failed calls so far
  ticket 60/64   76s elapsed   0 failed calls so far
128 model calls made (64 eval tickets x 2 variants), 0 failed after retries


In [ ]:
def accuracy(preds, golds):
    return sum(p == g for p, g in zip(preds, golds)) / len(golds)

def macro_f1(preds, golds):
    f1s = []
    for lab in LABELS:
        tp = sum(p == lab and g == lab for p, g in zip(preds, golds))
        fp = sum(p == lab and g != lab for p, g in zip(preds, golds))
        fn = sum(p != lab and g == lab for p, g in zip(preds, golds))
        f1s.append(2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0)
    return sum(f1s) / len(f1s)

def sign_test_p(b, c):
    """Two-sided exact sign test on the disagreements (b = few-shot only right, c = zero-shot only right)."""
    n = b + c
    if n == 0:
        return 1.0
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(min(b, c) + 1)) / 2 ** n)

if ROWS:
    gold = [r['gold'] for r in ROWS]
    majority = Counter(r['label'] for r in train).most_common(1)[0][0]
    table = [('zero-shot', [r['zero'] for r in ROWS]), (f'dynamic {K}-shot', [r['few'] for r in ROWS]),
             (f'kNN vote of the same {K} (no LLM)', [r['knn'] for r in ROWS]),
             (f'majority class ({majority})', [majority] * len(ROWS))]
    print(f'{"variant":36s} accuracy   macro-F1')
    for name, preds in table:
        print(f'{name:36s} {accuracy(preds, gold):7.1%}   {macro_f1(preds, gold):.3f}')

    print()
    for v, label in (('zero', 'zero-shot'), ('few', f'dynamic {K}-shot')):
        cs = [c for (vv, _), c in CALLS.items() if vv == v]
        inv = sum(1 for c in cs if c['pred'] == 'invalid')
        inexact = sum(1 for c in cs if c['pred'] != 'invalid' and not c['exact'])
        print(f'{label:16s} avg input tokens {sum(c["in_tok"] for c in cs) / len(cs):6.1f}   '
              f'avg output tokens {sum(c["out_tok"] for c in cs) / len(cs):4.1f}   '
              f'avg latency {sum(c["secs"] for c in cs) / len(cs):.2f}s   '
              f'unparseable {inv}   parsed-but-not-exact {inexact}   api errors {sum(1 for c in cs if c["error"])}')

    both = sum(r['zero'] == r['gold'] and r['few'] == r['gold'] for r in ROWS)
    few_only = sum(r['zero'] != r['gold'] and r['few'] == r['gold'] for r in ROWS)
    zero_only = sum(r['zero'] == r['gold'] and r['few'] != r['gold'] for r in ROWS)
    neither = len(ROWS) - both - few_only - zero_only
    print(f'\npaired: both right {both}, only few-shot right {few_only}, only zero-shot right {zero_only}, '
          f'both wrong {neither}')
    print(f'exact sign test on the {few_only + zero_only} disagreements: p = {sign_test_p(few_only, zero_only):.3f}  '
          f'(n = {len(ROWS)}, so treat small gaps as noise)')
else:
    print('skipped (no model calls were made)')

variant                              accuracy   macro-F1
zero-shot                              98.4%   0.984
dynamic 4-shot                        100.0%   1.000
kNN vote of the same 4 (no LLM)        95.3%   0.955
majority class (billing)               23.4%   0.076

zero-shot        avg input tokens   66.6   avg output tokens  5.3   avg latency 0.63s   unparseable 0   parsed-but-not-exact 0   api errors 0
dynamic 4-shot   avg input tokens  167.0   avg output tokens  5.3   avg latency 0.65s   unparseable 0   parsed-but-not-exact 0   api errors 0

paired: both right 63, only few-shot right 1, only zero-shot right 0, both wrong 0
exact sign test on the 1 disagreements: p = 1.000  (n = 64, so treat small gaps as noise)


In [ ]:
def slice_report(title, keyfn):
    print(title)
    print(f'  {"slice":38s} {"n":>3s}  {"zero":>6s}  {"few":>6s}  {"knn":>6s}')
    groups = defaultdict(list)
    for r in ROWS:
        groups[keyfn(r)].append(r)
    for name in sorted(groups):
        g = groups[name]
        z, f, k = (sum(r[v] == r['gold'] for r in g) / len(g) for v in ('zero', 'few', 'knn'))
        note = '   <-- few-shot underperforms' if f < z else ''
        print(f'  {str(name):38s} {len(g):3d}  {z:6.0%}  {f:6.0%}  {k:6.0%}{note}')
    print()

if ROWS:
    slice_report('by ticket kind:', lambda r: r['kind'])
    slice_report('by template:', lambda r: 'seen' if r['seen_template'] else 'held-out')
    slice_report('by gold label:', lambda r: r['gold'])
    slice_report('by retrieval purity:', lambda r: f'all {K} neighbors = gold' if all(l == r['gold'] for l in r['shot_labels']) else 'neighbors mixed/wrong')

    for title, cond in [('few-shot wrong where zero-shot was right', lambda r: r['few'] != r['gold'] == r['zero']),
                        ('few-shot right, zero-shot wrong', lambda r: r['few'] == r['gold'] != r['zero']),
                        ('kNN wrong (retrieval alone failed)', lambda r: r['knn'] != r['gold'])]:
        hits = [r for r in ROWS if cond(r)]
        print(f'--- {title}: {len(hits)} ---')
        for r in hits[:10]:
            print(f'ticket: {r["text"]!r} [{r["kind"]}, {"seen" if r["seen_template"] else "held-out"}]')
            print(f'   gold={r["gold"]} zero={r["zero"]} few={r["few"]} knn={r["knn"]}  shot labels={r["shot_labels"]}')
else:
    print('skipped (no model calls were made)')

by ticket kind:
  slice                                    n    zero     few     knn
  convention                               7    100%    100%     86%
  lookalike                                7     86%    100%     71%
  standard                                50    100%    100%    100%

by template:
  slice                                    n    zero     few     knn
  held-out                                24    100%    100%     88%
  seen                                    40     98%    100%    100%

by gold label:
  slice                                    n    zero     few     knn
  account_access                          11    100%    100%    100%
  billing                                 15    100%    100%     93%
  bug_report                              12    100%    100%    100%
  cancellation                            12    100%    100%    100%
  feature_request                         14     93%    100%     86%

by retrieval purity:
  slice                            

### Stress test (added after the first run)

The main eval came back near its ceiling: zero-shot got 98.4%, so there was almost no room for
few-shot to lose, and no slice showed it doing worse. That is a finding, but it doesn't tell me
what happens when the retrieved examples point the wrong way. So I added this check **after
seeing those results**, and it should be read that way: it is a post-hoc stress test, not part
of the 64-ticket eval, and it is not a fair sample of real tickets.

How it is built: new phrasings (none of these templates appear in train or eval) written to
sound like a *different* label, for example a feature request that talks about cancelling, or a
bug report about the billing page. I generate up to ~200, embed them, and keep only the ones
where the kNN vote of the 4 retrieved training examples is **wrong**. The selection uses the
gold label and retrieval only, before any model call. A separate random generator is used, so
the main dataset is untouched. By construction the retrieved examples mislead on every ticket
here. The question is whether few-shot copies them.


In [ ]:
# ---- build the stress set (retrieval + gold labels only; no model calls) ----
STRESS_TEMPLATES = [
    ('feature_request', "Could you add a way to downgrade my {plan} instead of cancelling it?"),
    ('feature_request', "Please add a refund request button to the {feature}"),
    ('feature_request', "I'd like the {feature} to show my billing history, can you add that?"),
    ('feature_request', "Please add {x} to the billing page"),
    ('feature_request', "Could you add a reminder before my {plan} renews?"),
    ('bug_report', "The billing page shows {error} when I open it"),
    ('bug_report', "The cancel button does nothing, it just shows {error}"),
    ('bug_report', "Downloading my invoice from the {feature} gives {error}"),
    ('bug_report', "The password reset page crashes with {error} on {device}"),
    ('billing', "I cancelled my {plan} {when} and was charged {amount} again, I want that money back"),
    ('billing', "Please refund the {amount} I was charged after I closed my account {since}"),
    ('account_access', "My password reset link expired before I could use it on {device}"),
    ('account_access', "I forgot my password and the reset page says my email is not recognized"),
    ('cancellation', "I'm done with the {plan}, please don't renew it or bill me again"),
    ('cancellation', "Please cancel before the renewal charge of {amount} goes through"),
]
srng = random.Random(SEED + 1)          # separate generator: the main dataset and its fingerprint are untouched

def stress_cores(tmpl):
    names = list(dict.fromkeys(re.findall(r'\{(\w+)\}', tmpl)))
    out = [tmpl.format(**dict(zip(names, c))) for c in itertools.product(*[SLOT[n] for n in names])]
    srng.shuffle(out)
    return out

def stress_decorate(core):
    o, t = srng.choice(OPEN), srng.choice(TAIL)
    if o and not re.match(r"I\b", core):
        core = core[0].lower() + core[1:]
    return o + core + t

known = {norm_key(r['text']) for r in train + eval_}
pool = []
for si, (lab, tmpl) in enumerate(STRESS_TEMPLATES):
    for core in stress_cores(tmpl)[:15]:
        text = stress_decorate(core)
        if norm_key(text) not in known:
            known.add(norm_key(text))
            pool.append({'text': text, 'label': lab, 'template': f's{si + 1:02d}'})
for p_, (ids, sims) in zip(pool, retrieve(embed([p_['text'] for p_ in pool]), K)):
    p_['ids'], p_['knn'] = ids, knn_vote(ids)
    p_['n_gold'] = sum(train[j]['label'] == p_['label'] for j in ids)      # neighbors carrying the right label
mislead = [p_ for p_ in pool if p_['knn'] != p_['label']]

by_t = defaultdict(list)
for p_ in mislead:
    by_t[p_['template']].append(p_)
STRESS = []
while len(STRESS) < 30 and any(by_t.values()):
    for t_ in sorted(by_t):
        if by_t[t_] and len(STRESS) < 30:
            STRESS.append(by_t[t_].pop(0))

with open('data/stress.jsonl', 'w') as f:
    for p_ in STRESS:
        f.write(json.dumps({k: p_[k] for k in ('text', 'label', 'template')}, sort_keys=True) + '\n')
print(f'candidates {len(pool)}  ->  retrieval misled the kNN vote on {len(mislead)}  ->  kept {len(STRESS)} (cap 30)')
print('kept by gold label:', dict(Counter(p_['label'] for p_ in STRESS)))
print('kept by template:  ', dict(sorted(Counter(p_['template'] for p_ in STRESS).items())))
print('neighbors (of 4) carrying the gold label:', dict(sorted(Counter(p_['n_gold'] for p_ in STRESS).items())))
if len(STRESS) < 10:
    print('WARNING: fewer than 10 stress tickets -- too few to say anything; report that instead of the numbers.')

candidates 120  ->  retrieval misled the kNN vote on 32  ->  kept 30 (cap 30)
kept by gold label: {'feature_request': 15, 'bug_report': 13, 'cancellation': 2}
kept by template:   {'s01': 5, 's02': 4, 's03': 1, 's05': 5, 's07': 2, 's09': 11, 's15': 2}
neighbors (of 4) carrying the gold label: {0: 16, 1: 12, 2: 2}


In [ ]:
# ---- run zero-shot vs few-shot on the stress set (about 60 calls) ----
STRESS_ROWS = []
if API_OK and STRESS:
    for p_ in STRESS:
        shots = [train[j] for j in p_['ids']]
        z = classify('zero', p_['text'])
        f = classify('few', p_['text'], shots)
        if z['error'] or f['error']:
            continue
        STRESS_ROWS.append({**p_, 'zero': z['pred'], 'few': f['pred'],
                            'shot_labels': [train[j]['label'] for j in p_['ids']],
                            'shot_texts': [train[j]['text'] for j in p_['ids']]})
    n = len(STRESS_ROWS)
    if n < len(STRESS):
        print(f'WARNING: only {n}/{len(STRESS)} stress tickets scored -- re-run this cell to retry before reading numbers.')
    if n:
        g = [r['label'] for r in STRESS_ROWS]
        print(f'stress set (n = {n}; the kNN vote is wrong on all of them by construction)')
        print(f'  zero-shot accuracy      {accuracy([r["zero"] for r in STRESS_ROWS], g):.1%}')
        print(f'  dynamic {K}-shot accuracy {accuracy([r["few"] for r in STRESS_ROWS], g):.1%}')
        few_only = sum(r['zero'] != r['label'] and r['few'] == r['label'] for r in STRESS_ROWS)
        zero_only = sum(r['zero'] == r['label'] and r['few'] != r['label'] for r in STRESS_ROWS)
        print(f'  only few-shot right {few_only}, only zero-shot right {zero_only}, '
              f'sign test p = {sign_test_p(few_only, zero_only):.3f}')
        copied = sum(r['few'] == r['knn'] for r in STRESS_ROWS)
        print(f"  few-shot answered with the neighbors' (wrong) vote on {copied}/{n} tickets")

        print()
        for title, keyfn in (('by gold label', lambda r: r['label']),
                             ('by how many of the 4 neighbors carried the right label', lambda r: r['n_gold'])):
            print(title + ':')
            groups = defaultdict(list)
            for r in STRESS_ROWS:
                groups[keyfn(r)].append(r)
            for name in sorted(groups):
                gg = groups[name]
                z_ = sum(r['zero'] == r['label'] for r in gg) / len(gg)
                f_ = sum(r['few'] == r['label'] for r in gg) / len(gg)
                print(f'  {str(name):18s} n={len(gg):3d}  zero {z_:5.0%}  few {f_:5.0%}   '
                      f'{"<-- few-shot underperforms" if f_ < z_ else ""}')
            print()

        bad = [r for r in STRESS_ROWS if r['few'] != r['label'] and r['zero'] == r['label']]
        print(f'--- few-shot wrong where zero-shot was right: {len(bad)} ---')
        for r in bad[:10]:
            print(f'ticket: {r["text"]!r}')
            print(f'   gold={r["label"]}  zero-shot={r["zero"]}  few-shot={r["few"]}  neighbor vote={r["knn"]}')
            for t_, l_ in zip(r['shot_texts'], r['shot_labels']):
                print(f'     shot: {t_!r} -> {l_}')
        both_wrong = [r for r in STRESS_ROWS if r['few'] != r['label'] and r['zero'] != r['label']]
        print(f'\n--- both wrong: {len(both_wrong)} ---')
        for r in both_wrong[:6]:
            print(f'ticket: {r["text"]!r}   gold={r["label"]}  zero-shot={r["zero"]}  few-shot={r["few"]}')
else:
    print('skipped (no API access, or no stress tickets)')

stress set (n = 30; the kNN vote is wrong on all of them by construction)
  zero-shot accuracy      100.0%
  dynamic 4-shot accuracy 100.0%
  only few-shot right 0, only zero-shot right 0, sign test p = 1.000
  few-shot answered with the neighbors' (wrong) vote on 0/30 tickets

by gold label:
  bug_report         n= 13  zero  100%  few  100%   
  cancellation       n=  2  zero  100%  few  100%   
  feature_request    n= 15  zero  100%  few  100%   

by how many of the 4 neighbors carried the right label:
  0                  n= 16  zero  100%  few  100%   
  1                  n= 12  zero  100%  few  100%   
  2                  n=  2  zero  100%  few  100%   

--- few-shot wrong where zero-shot was right: 0 ---

--- both wrong: 0 ---


**Stress-test results.**

This was added after the main results and is deliberately adversarial, so read it that way. I
generated 120 new tickets; retrieval misled the kNN vote on 32 of them, and I kept 30. For 16 of
the 30, none of the 4 retrieved examples carried the right label; for 12, one did; for 2, two did.

| | accuracy (n = 30) |
|---|---|
| zero-shot | 100% |
| dynamic 4-shot | 100% |
| kNN vote (wrong on all 30 by construction) | 0% |

Few-shot never gave the neighbors' wrong vote (0 of 30), and there was no ticket where it lost to
zero-shot. So even when every retrieved example pointed the wrong way, Haiku answered from the
ticket. I did not find a failure case this way either.

Limits: 30 tickets from 7 of the 15 phrasings I wrote, and one of them (a password-reset-page
crash) made up 11 of the 30. Only `feature_request`, `bug_report` and `cancellation` made it in;
retrieval was not fooled on the billing or account-access phrasings I wrote. These are short,
unambiguous tickets that I wrote myself, which is about the easiest case for a strong model. This
shows few-shot was not fragile here. It does not show it is robust in general.


**Results.**

| | accuracy | macro-F1 | input tokens/call | latency |
|---|---|---|---|---|
| zero-shot | 98.4% | 0.984 | 66.6 | 0.63s |
| dynamic 4-shot | 100% | 1.000 | 167.0 | 0.65s |
| kNN vote of the same 4 (no LLM) | 95.3% | 0.955 | none | none |
| majority class | 23.4% | 0.076 | none | none |

Out of 64 tickets, 63 were right with both prompts, 1 was right only with few-shot, and none
were right only with zero-shot. The sign test on that one disagreement gives p = 1.0, so this is
not a measurable difference. Few-shot sends about 100 more input tokens per ticket (2.5x) and
is about as fast (0.65s vs 0.63s).

**No slice where few-shot lost.** I looked by ticket kind, template, gold label, and retrieval
quality. Zero-shot never beat few-shot on any of them. The reason is that this eval is near its
ceiling: zero-shot already gets 98.4%, so there is almost nothing for few-shot to fix or break.
I tried to break it with the stress test above (30 tickets where retrieval pointed the wrong
way) and it still didn't lose. So I never found a slice where few-shot underperformed zero-shot.

The closest thing to a failure in the main eval is the retrieval-only baseline, which got 3
tickets wrong. All 3 are held-out-template tickets, such as "add an option to pause billing
instead of cancelling my plan". Their nearest neighbors were mostly `cancellation` tickets,
because retrieval matches on topic words like "cancel" and "billing", not on what the customer is
asking for. Yet few-shot got all 3 right, even when all 4 retrieved examples were labeled
`cancellation`. So at least here the model read the ticket and did not copy its examples. The one
win for few-shot was "Add a way to reset my password from the export": zero-shot called it
`account_access`, while 3 of the 4 retrieved examples were `feature_request`.

What this doesn't show: it's n = 64 on synthetic templates with a strong model, the hard slices
have only 7 tickets each (one ticket is 14 points), and I didn't test a zero-shot prompt that
spells out the labeling rule, which is the obvious cheaper rival.


## Part 4: Decision memo

**Recommendation: ship prompt-only (zero-shot) with Haiku now. Keep dynamic few-shot as the cheap
next step, and don't fine-tune.**

| option | accuracy / macro-F1 | cost per ticket | what it needs to keep working |
|---|---|---|---|
| do nothing (majority class, or people triage) | 23.4% / 0.076 | people reading every ticket | staff time |
| prompt only (zero-shot) | 98.4% / 0.984 | 66.6 input tokens, 0.63s | just the prompt |
| retrieval only (kNN vote, no LLM) | 95.3% / 0.955 | one embedding lookup, no model call | the labeled set + embedder + index |
| dynamic few-shot | 100% / 1.000 | 167.0 input tokens, 0.65s | labeled set + embedder + index |
| fine-tuning | not run | training + hosting | labeled set, a training pipeline, retraining when labels change |

**Why zero-shot first.** It already gets 98.4%. Dynamic few-shot fixed one ticket out of 64
(sign test p = 1.0), cost about 100 more input tokens each (2.5x), and was no faster. It also
needs a labeled set, an embedder and an index kept up to date. I could not make it hurt, either:
in the main eval it never lost a ticket, and in the stress test it ignored misleading examples on
30 of 30. That is why I'd call it the low-risk upgrade and not something to avoid. The one
place it helped was a lookalike ticket ("add a way to reset my password from the export") that
zero-shot sent to `account_access`.

**Why not retrieval alone.** It costs nothing per call, but it dropped to 71% on lookalike
tickets and 88% on held-out phrasings, because embeddings match topic words ("cancel", "billing")
and not what the customer is asking for. Those are the tickets that matter. It is a fallback if
the API is down, not the main path.

**When fine-tuning would be worth it.** Nothing in this run points to it. The gap left to close is
at most 1.6 points. I'd revisit if:
- *the error rate on real tickets stays high.* If prompt plus few-shot still misses more than a
  few percent on a real labeled sample, and each miss is costly, there is a gap worth paying to
  close. Here there isn't one.
- *volume makes tokens the bill.* Few-shot adds about 100 input tokens per ticket, so about
  100,000 extra per 1,000 tickets. Fine-tuning pays off only if (training + hosting + retraining)
  is less than the tokens a shorter prompt saves at my volume. I haven't priced it and don't know
  the volume, so I can't claim it does.
- *the labeling rules are stable.* Fine-tuning bakes them into the weights. The charge-versus-
  cancel convention here is exactly the kind of rule that gets revised, and a prompt edit is
  cheaper than a retrain.
- *I can't send tickets to an API,* for privacy or latency, and need a small self-hosted model.
  That is a constraint, not a result from this run.

**How much to trust this.** The tickets are synthetic and templated, short and unambiguous, and
Haiku is a strong model. The eval was near its ceiling, so it cannot separate zero-shot from
few-shot at n = 64. Before making the decision for real, I'd re-run this on a couple hundred
real, messier tickets.
